# PatentIntel.AI - 100% Free GPU Model Fine-Tuning (Google Colab & Kaggle)
**Developer**: Madhuaravind P (`madhuaravind21`)

This notebook fine-tunes Meta Llama 3.1 8B / Qwen 2.5 on your master patent dataset for 100% FREE on Google Colab or Kaggle GPUs and automatically pushes the trained model to your Hugging Face account: `madhuaravind21/patentintel-llama3-1m`.

In [ ]:
# 1. Install Unsloth, PyTorch & Hugging Face Libraries
!pip install --quiet unsloth torch transformers datasets trl peft bitsandbytes huggingface_hub

In [ ]:
# 2. Authenticate with Hugging Face Hub
from huggingface_hub import login
import getpass

print("Enter your Hugging Face Write Token (from huggingface.co/settings/tokens):")
hf_token = getpass.getpass()
login(token=hf_token)

In [ ]:
# 3. Load Base Model & Unsloth GPU Acceleration
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 2048
model_name = "unsloth/llama-3.1-8b-instruct-bnb-4bit"

print(f"Loading {model_name} onto GPU...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# 4. Stream & Format Patent Training Dataset
dataset_name = "madhuaravind21/patentintel-million-dataset"
print("Loading patent training dataset...")

try:
    dataset = load_dataset(dataset_name, split="train")
except Exception as e:
    print("Fallback to direct synthetic patent dataset loading...")
    dataset = load_dataset("json", data_files={"train": "patentintel_master_dataset.json"})["train"]

prompt_template = "### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n{}"

def format_prompts(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        texts.append(prompt_template.format(inst, inp, out) + tokenizer.eos_token)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

In [ ]:
# 5. Execute Fine-Tuning & Export Trained Model
output_repo = "madhuaravind21/patentintel-llama3-1m"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        output_dir="./results",
    ),
)

print("Starting Free GPU Fine-Tuning Process...")
trainer.train()

print(f"Pushing fine-tuned AI model to Hugging Face Hub: https://huggingface.co/{output_repo}")
model.push_to_hub_merged(output_repo, tokenizer, save_method="merged_16bit", token=hf_token)
print("SUCCESS! Your custom patent AI model is live and ready to connect in PatentIntel.AI Settings!")